# hook_style — analysis

Question vs. statement hook line, measured as **stayed-to-watch %** at 7 days.
Everything here is fixed by [`PREREGISTRATION.md`](PREREGISTRATION.md); this notebook only
applies it. Run it **once**, when the stop rule is reached (26 measured clips per arm, or
26 weeks after launch) — there are no interim looks at the treatment effect.

Until real outcomes exist it renders the honest empty state, not a placeholder result.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from clipper import experiment as ex
from clipper import experiment_analysis as ea
from clipper.paths import REPO_ROOT

EXP_DIR = Path(os.environ.get("EXPERIMENT_DIR", REPO_ROOT / "experiments" / "hook_style"))
cfg = ex.load_config(EXP_DIR / "config.yaml")
print(f"experiment: {cfg.name}   status: {cfg.status}   launch: {cfg.launch_date}")

## 1. Integrity

The assignment log is recomputed from the seed and the pre-registration/design hashes are checked. Analysis stops here if anything fails.

In [ ]:
problems = ex.verify(cfg)
assert not problems, "assignment log failed verification:\n" + "\n".join(problems)
print(f"OK — {len(ex.read_assignments(cfg))} assignment(s) match the seed; pre-registration and design unchanged")

## 2. Funnel

Enrolment happens at the selection call, so most assigned clips are rejected before they are ever published. Each stage is shown by arm.

In [ ]:
rows = ea.build_rows(cfg, ea.load_outcomes(EXP_DIR / "outcomes.csv"))
res = ea.analyze(cfg, rows)
arms = [res["control"], res["treatment"]]
pd.DataFrame(res["funnel"]).T.loc[arms]

## 3. Decision

Produced by the pre-registered rule, not by reading the chart.

In [ ]:
d = res["decision"]
print(d["label"])
for f in d["flags"]:
    print("  flag:", f)

## 4. Primary result

Each dot is one published clip; the bar is the arm mean. The title carries the difference and its 95% bootstrap interval.

In [ ]:
ax = ea.plot_primary(res, rows)
if ax is None:
    print("No primary result yet: fewer than 2 measured clips in an arm.")
else:
    plt.tight_layout()
    plt.show()

In [ ]:
if "primary" in res:
    p = res["primary"]
    pd.DataFrame(
        {
            "Welch": [p["welch"]["diff"], p["welch"]["p"]],
            "Mann-Whitney": [None, p["mann_whitney"]["p"]],
            "Bootstrap 95% CI": [f"[{p['bootstrap']['lo']:+.1f}, {p['bootstrap']['hi']:+.1f}]", None],
        },
        index=["difference (pts) / CI", "p-value"],
    )

## 5. Diagnostics, sensitivity, secondary outcomes

Any flag here downgrades a ship/keep decision to inconclusive; a sample-ratio mismatch withholds it.

In [ ]:
print(ea.render_memo(res))

## 6. Decision memo

`uv run clipper experiment analyze` writes this same memo to `decision_memo.md` and the numbers to `results.json`. The *what to test next* section is for a human to write.